In [1]:
#Este notebook lo creamos para traer valores contables de los tickers que necesitamos y poder armar un P&L
# 1) Importamos las librerias que vamos a utilizar para 

import yfinance as yf
import pandas as pd
from datetime import datetime

In [2]:
# 2) Creamos una lista de tickers que de momento vamos a traer para poder analizar los valores del P&L
# Esta lista viene del archivo "DIM_Tickers.csv" que se crea en otro notebook, es importante correr ese notebook si se agregaron tickers. 

df_tickers = pd.read_csv("DIM_Tickers.csv")

# Filtramos solo acciones (equities)
df_equities = df_tickers[df_tickers["QuoteType"] == "EQUITY"]

# Armamos la lista final de tickers
lista_tickers = df_equities["TickerYahoo"].tolist()

print(lista_tickers)

['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA', 'ORCL', 'PLTR', 'RDDT', 'MU', 'AMD', 'TSM', 'ALAB', 'MSTR', 'QCOM', 'TSM', 'YPF', 'PEP', 'UNH', 'VIST']


In [3]:
#Creamos una función para obtener los valores de P&L

def get_income_statement(ticker, frequency="annual"):
    """
    frequency: 'annual' or 'quarterly'
    """
    t = yf.Ticker(ticker)

    if frequency == "annual":
        df = t.financials
    else:
        df = t.quarterly_financials

    if df is None or df.empty:
        return pd.DataFrame()

    # Transponemos
    df = df.T

    # El index ahora son los períodos (fechas)
    df.index.name = "Period"

    df = df.reset_index()

    # Pasamos a formato largo
    df = df.melt(
        id_vars="Period",
        var_name="Metric",
        value_name="Value"
    )

    df["Ticker"] = ticker
    df["Frequency"] = frequency

    return df

In [4]:
#Creamos un bucle para pasarle los tickers y obtener los valores 
dfs = []

for ticker in lista_tickers:
    print(f"Procesando {ticker}...")
    dfs.append(get_income_statement(ticker, "annual"))
    dfs.append(get_income_statement(ticker, "quarterly"))

pnl_raw = pd.concat(dfs, ignore_index=True)

Procesando AAPL...
Procesando MSFT...
Procesando GOOGL...
Procesando AMZN...
Procesando META...
Procesando NVDA...
Procesando TSLA...
Procesando ORCL...
Procesando PLTR...
Procesando RDDT...
Procesando MU...
Procesando AMD...
Procesando TSM...
Procesando ALAB...
Procesando MSTR...
Procesando QCOM...
Procesando TSM...
Procesando YPF...
Procesando PEP...
Procesando UNH...
Procesando VIST...


In [5]:
pnl_raw

,Period,Metric,Value,Ticker,Frequency
0,2025-09-30,Tax Effect Of Unusual Items,0.0,AAPL,annual
1,2024-09-30,Tax Effect Of Unusual Items,0.0,AAPL,annual
2,2023-09-30,Tax Effect Of Unusual Items,0.0,AAPL,annual
3,2022-09-30,Tax Effect Of Unusual Items,0.0,AAPL,annual
4,2021-09-30,Tax Effect Of Unusual Items,NaN,AAPL,annual
...,...,...,...,...,...
10316,2025-09-30,Operating Revenue,706135000.0,VIST,quarterly
10317,2025-06-30,Operating Revenue,610542000.0,VIST,quarterly
10318,2025-03-31,Operating Revenue,438456000.0,VIST,quarterly
10319,2024-12-31,Operating Revenue,471318000.0,VIST,quarterly


In [6]:
#Consideramos unicamente las métricas del P&L y filtrarlas

metrics_pnl = [
    "Total Revenue",
    "Cost Of Revenue",
    "Gross Profit",
    "Operating Expense",
    "Operating Income",
    "EBITDA",
    "Net Income",
    "Basic EPS"
]

pnl = pnl_raw[pnl_raw["Metric"].isin(metrics_pnl)].copy()

In [7]:
pnl["Period"] = pd.to_datetime(pnl["Period"])
pnl = pnl.sort_values(["Ticker", "Period", "Metric"])

In [8]:
pnl.to_csv("income_statement_yahoo.csv", index=False)